# Fila Digital — Data Science (GS CitizenTech 2026.2)

**Problema:** quem precisa de atendimento público de urgência não sabe quanto vai esperar em cada unidade, e as unidades ficam desequilibradas (uma lotada, outra vazia).

**Pergunta:** dado o estado atual das filas, qual será o tempo de espera de cada setor daqui a ~15 minutos, e qual a chance de ele passar de um limite "longo"?

**Fonte dos dados:** tempo médio de espera dos Prontos-Atendimentos de Barueri/SP, publicado em tempo real pela Prefeitura
(https://saudeservicos.barueri.sp.gov.br/AgendamentoOnline/TempoAtendimentoPS.aspx). Dados públicos e agregados, sem nenhum dado pessoal.
Cada leitura da página vira linhas `local, tempo_espera, ts`. Coletamos leituras ao longo do tempo para formar a série.

**Modelos (exigidos pela disciplina):**
- Regressão linear → prever `espera_futura` (minutos, variável numérica)
- Regressão logística → prever `espera_longa` (1 se `espera_futura >= LIMIAR_LONGA`, evento binário)

**Modos de execução:** `MODO = "real"` usa o CSV coletado da página. `MODO = "simulado"` gera dados sintéticos *só para testar o pipeline* (e fica marcado como `SIMULADO` no CSV). A entrega final deve usar `real`.

In [ ]:
import re, time, unicodedata
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             confusion_matrix, ConfusionMatrixDisplay,
                             precision_score, recall_score, f1_score, roc_auc_score)

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

## 1. Configuração

In [ ]:
MODO = "simulado"          # troque para "real" depois de coletar dados (seção 2)
URL = "https://saudeservicos.barueri.sp.gov.br/AgendamentoOnline/TempoAtendimentoPS.aspx"
HEADERS = {"User-Agent": "Mozilla/5.0 (projeto academico FIAP - GS CitizenTech)"}

# No Colab, para não perder o CSV quando a sessão cair, monte o Drive e aponte para ele:
# from google.colab import drive; drive.mount('/content/drive')
# CSV_REAL = "/content/drive/MyDrive/gs_citizentech/espera_barueri.csv"
CSV_REAL = "espera_barueri.csv"

HORIZONTE_MIN = 15         # prever a espera daqui a quantos minutos
TOLERANCIA_MIN = 7         # folga para casar leituras no tempo
LIMIAR_LONGA = 20          # minutos: acima disso consideramos espera "longa"
PROP_TREINO = 0.8          # split temporal (80% mais antigo = treino)

## 2. Coleta dos dados reais (API pública da Prefeitura de Barueri)

Rode `coletar_continuamente(...)` por algumas horas (idealmente em horários e dias diferentes: manhã, tarde, noite, fim de semana). Cada leitura é salva no CSV, sem duplicar.
Quanto mais variedade de horário, melhor o modelo aprende o padrão de pico.

In [ ]:
def _parse_minutos(t):
    """"2 minutos" -> 2 | "1 h e 36 min" -> 96 | "Sem espera" -> 0."""
    t = str(t).lower()
    if "sem " in t:
        return 0.0
    h = re.search(r"(\d+)\s*h", t)
    m = re.search(r"(\d+)\s*min", t)
    if h or m:
        return (int(h.group(1)) * 60 if h else 0) + (int(m.group(1)) if m else 0)
    n = re.search(r"(\d+)", t)
    return float(n.group(1)) if n else np.nan


def coletar_snapshot():
    """Lê a página e devolve um DataFrame com uma linha por local/setor."""
    r = requests.get(URL, timeout=30, headers=HEADERS)
    r.raise_for_status()
    tabelas = pd.read_html(StringIO(r.text))
    tab = next((t for t in tabelas if t.shape[1] == 3), None)
    if tab is None:
        raise ValueError("Tabela com 3 colunas não encontrada. O layout da página mudou?")
    tab.columns = ["local", "tempo_txt", "atualizado_em"]
    tab = tab.dropna(subset=["local", "tempo_txt"]).copy()
    txt = tab["tempo_txt"].astype(str).str.lower()
    num = txt.map(_parse_minutos).astype(float)
    tab["tempo_espera"] = num
    tab = tab.dropna(subset=["tempo_espera"])
    tab["ts"] = pd.to_datetime(tab["atualizado_em"], dayfirst=True, errors="coerce")
    tab = tab.dropna(subset=["ts"])
    tab["coletado_em"] = pd.Timestamp.now()
    tab["fonte"] = "Prefeitura de Barueri - Tempo de espera PS"
    return tab[["local", "tempo_espera", "ts", "coletado_em", "fonte"]]


def salvar_snapshot(novo, caminho=CSV_REAL):
    p = Path(caminho)
    if p.exists():
        antigo = pd.read_csv(p, parse_dates=["ts", "coletado_em"])
        novo = pd.concat([antigo, novo])
    novo = novo.drop_duplicates(["local", "ts"]).sort_values(["ts", "local"])
    novo.to_csv(p, index=False)
    return len(novo)


def coletar_continuamente(duracao_min=120, intervalo_min=5):
    fim = time.time() + duracao_min * 60
    while time.time() < fim:
        try:
            total = salvar_snapshot(coletar_snapshot())
            print(f"{pd.Timestamp.now():%d/%m %H:%M} ok - {total} linhas no CSV")
        except Exception as e:
            print("falha na coleta:", e)
        time.sleep(intervalo_min * 60)

# Descomente para coletar (mantenha a aba do Colab aberta):
# coletar_continuamente(duracao_min=180, intervalo_min=5)

## 3. Dados simulados (somente para testar o pipeline)

Usado quando `MODO = "simulado"`. Imita os padrões que esperamos ver (pico de manhã e à noite, setores com espera maior, persistência no tempo, picos pontuais).
**Não use esses dados na entrega final.**

In [ ]:
LOCAIS_EXEMPLO = [
    "PRONTO ATENDIMENTO - ENGENHO NOVO - ATENDIMENTO", "PRONTO ATENDIMENTO - ENGENHO NOVO - TRIAGEM",
    "PRONTO ATENDIMENTO - ENGENHO NOVO- CLINICO GERAL", "PRONTO ATENDIMENTO - ENGENHO NOVO- ORTOPEDIA",
    "PRONTO ATENDIMENTO - ENGENHO NOVO- PEDIATRIA",
    "PRONTO ATENDIMENTO - IMPERIAL - ATENDIMENTO", "PRONTO ATENDIMENTO - IMPERIAL - TRIAGEM",
    "PRONTO ATENDIMENTO - IMPERIAL - CLINICO GERAL", "PRONTO ATENDIMENTO - IMPERIAL - ORTOPEDIA",
    "PRONTO ATENDIMENTO - IMPERIAL - PEDIATRIA",
    "PRONTO ATENDIMENTO CAMARGO - ATENDIMENTO", "PRONTO ATENDIMENTO CAMARGO - TRIAGEM",
    "PRONTO ATENDIMENTO CAMARGO- EMERGÊNCIA", "PRONTO ATENDIMENTO CAMARGO- ORTOPEDIA",
    "PRONTO ATENDIMENTO CAMARGO- PEDIATRIA",
    "PRONTO ATENDIMENTO ADULTO - PAA - ATENDIMENTO", "PRONTO ATENDIMENTO ADULTO - PAA - TRIAGEM",
    "PRONTO ATENDIMENTO ADULTO - PAA- CLINICO - SINTOMAS RESPIRATORIOS",
    "PRONTO ATENDIMENTO ADULTO - PAA- CLINICO GERAL", "PRONTO ATENDIMENTO ADULTO - PAA- ORTOPEDIA",
    "PRONTO ATENDIMENTO INFANTIL - PAI - ATENDIMENTO", "PRONTO ATENDIMENTO INFANTIL - PAI- PEDIATRIA",
    "PRONTO ATENDIMENTO MUTINGA - TRIAGEM", "PRONTO ATENDIMENTO MUTINGA- CLINICO GERAL",
    "PRONTO ATENDIMENTO MUTINGA- PEDIATRIA",
]

def gerar_simulado(dias=10, passo_min=15, seed=42):
    rng = np.random.default_rng(seed)
    idx = pd.date_range("2026-10-26 00:00", periods=dias * 24 * 60 // passo_min, freq=f"{passo_min}min")
    horas = idx.hour + idx.minute / 60
    perfil = 0.35 + 1.0 * np.exp(-((horas - 10) / 2.5) ** 2) + 0.9 * np.exp(-((horas - 19) / 2.5) ** 2)
    fds = (idx.dayofweek >= 5).astype(float)
    base_setor = {"ATENDIMENTO": 5, "TRIAGEM": 8, "CLINICO": 18, "ORTOPEDIA": 15, "PEDIATRIA": 12, "EMERGENCIA": 6}
    base_unid = {"PAA": 1.5, "CAMARGO": 1.1, "IMPERIAL": 0.9}
    linhas = []
    for local in LOCAIS_EXEMPLO:
        s, u = extrair_setor(local), extrair_unidade(local)
        nivel = base_setor.get(s, 10) * base_unid.get(u, 1.0)
        alvo = nivel * perfil * (1 + 0.25 * fds)
        x = np.zeros(len(idx)); x[0] = alvo[0]
        surto = 0.0
        for t in range(1, len(idx)):
            surto = surto * 0.8 + (rng.random() < 0.012) * rng.uniform(15, 40)
            x[t] = 0.8 * x[t - 1] + 0.2 * alvo[t] + rng.normal(0, 1.2) + surto * 0.3
        espera = np.clip(np.round(x), 0, None)
        linhas.append(pd.DataFrame({"local": local, "tempo_espera": espera, "ts": idx,
                                    "coletado_em": idx, "fonte": "SIMULADO"}))
    return pd.concat(linhas, ignore_index=True)

## 4. Carregamento e preparação

In [ ]:
def _norm(s):
    return unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().upper()

def extrair_unidade(local):
    n = _norm(local)
    for chave in ["ENGENHO NOVO", "IMPERIAL", "CAMARGO", "MUTINGA", "HMB"]:
        if chave in n:
            return chave
    if re.search(r"\bPAA\b", n): return "PAA"
    if re.search(r"\bPAI\b", n): return "PAI"
    return "OUTRA"

def extrair_setor(local):
    n = _norm(local).replace("PRONTO ATENDIMENTO", "")
    for chave in ["TRIAGEM", "EMERGENCIA", "ORTOPEDIA", "PEDIATRIA", "GINECOLOGIA", "ODONTOLOGIA", "CLINICO"]:
        if chave in n:
            return chave
    return "ATENDIMENTO" if "ATENDIMENTO" in n else "OUTRO"

def carregar():
    if MODO == "real":
        if not Path(CSV_REAL).exists():
            raise FileNotFoundError(f"{CSV_REAL} não existe. Rode a coleta (seção 2) ou suba o CSV.")
        df = pd.read_csv(CSV_REAL, parse_dates=["ts"])
    else:
        print("ATENÇÃO: MODO simulado. Não use esses dados na entrega final.")
        df = gerar_simulado()
        df.to_csv("espera_simulado.csv", index=False)
    return df.drop_duplicates(["local", "ts"]).sort_values(["ts", "local"]).reset_index(drop=True)

df = carregar()
df["unidade"] = df["local"].map(extrair_unidade)
df["setor"] = df["local"].map(extrair_setor)
df["sintomas"] = df["local"].map(lambda s: int(bool(re.search("SINTOMAS|SUSPEITA", _norm(s)))))
print(df.shape, "| de", df["ts"].min(), "até", df["ts"].max())
print("fonte:", df["fonte"].unique())
print("valores ausentes:\n", df.isna().sum()[lambda s: s > 0])
df.head()

In [ ]:
# Exploração rápida
df["hora"] = df["ts"].dt.hour
resumo = df.groupby("setor")["tempo_espera"].agg(["count", "mean", "median", "max"]).round(1)
print(resumo.sort_values("mean", ascending=False))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df.groupby("hora")["tempo_espera"].mean().plot(ax=ax[0], marker="o")
ax[0].set_title("Espera média por hora do dia"); ax[0].set_xlabel("hora"); ax[0].set_ylabel("minutos")
df.groupby("setor")["tempo_espera"].mean().sort_values().plot.barh(ax=ax[1])
ax[1].set_title("Espera média por setor"); ax[1].set_xlabel("minutos")
plt.tight_layout(); plt.show()

### 4.1 Construção das variáveis (features) e do alvo

- `espera_atual`: tempo informado agora.
- `espera_passada`: tempo informado ~`HORIZONTE_MIN` minutos atrás (dá a tendência).
- `media_geral`, `media_unidade`, `n_alta`: carga das outras filas **no mesmo instante** (usa só informação do presente).
- `hora_sen/hora_cos`, `fim_de_semana`: padrão de horário.
- **Alvo:** `espera_futura` = tempo do mesmo local ~`HORIZONTE_MIN` minutos depois. `espera_longa` = `espera_futura >= LIMIAR_LONGA`.

In [ ]:
def adicionar_features(d):
    d = d.sort_values("ts").copy()
    h = d["ts"].dt.hour + d["ts"].dt.minute / 60
    d["hora_sen"] = np.sin(2 * np.pi * h / 24)
    d["hora_cos"] = np.cos(2 * np.pi * h / 24)
    d["fim_de_semana"] = (d["ts"].dt.dayofweek >= 5).astype(int)
    d["espera_atual"] = d["tempo_espera"]
    d["media_geral"] = d.groupby("ts")["tempo_espera"].transform("mean")
    d["media_unidade"] = d.groupby(["ts", "unidade"])["tempo_espera"].transform("mean")
    d["n_alta"] = d.groupby("ts")["tempo_espera"].transform(lambda s: (s >= LIMIAR_LONGA).sum())

    # leitura de ~H minutos atrás do mesmo local
    passado = d[["local", "ts", "tempo_espera"]].copy()
    passado["ts"] = passado["ts"] + pd.Timedelta(minutes=HORIZONTE_MIN)
    passado = passado.rename(columns={"tempo_espera": "espera_passada"}).sort_values("ts")
    d = pd.merge_asof(d.sort_values("ts"), passado, on="ts", by="local",
                      direction="nearest", tolerance=pd.Timedelta(minutes=TOLERANCIA_MIN))
    d["tendencia"] = d["espera_atual"] - d["espera_passada"]
    return d

def adicionar_alvo(d):
    futuro = d[["local", "ts", "tempo_espera"]].copy()
    futuro["ts"] = futuro["ts"] - pd.Timedelta(minutes=HORIZONTE_MIN)
    futuro = futuro.rename(columns={"tempo_espera": "espera_futura"}).sort_values("ts")
    d = pd.merge_asof(d.sort_values("ts"), futuro, on="ts", by="local",
                      direction="nearest", tolerance=pd.Timedelta(minutes=TOLERANCIA_MIN))
    d["espera_longa"] = (d["espera_futura"] >= LIMIAR_LONGA).astype(int)
    return d

feats = adicionar_features(df)
base = adicionar_alvo(feats).dropna(subset=["espera_passada", "espera_futura"]).reset_index(drop=True)

NUM = ["espera_atual", "espera_passada", "tendencia", "media_geral", "media_unidade",
       "n_alta", "hora_sen", "hora_cos", "fim_de_semana", "sintomas"]
CAT = ["unidade", "setor"]
print("linhas para modelagem:", len(base))
print("proporção de espera longa:", round(base["espera_longa"].mean(), 3))
base[NUM + CAT + ["espera_futura", "espera_longa"]].head()

## 5. Divisão treino/teste (sem vazamento)

Dados em série temporal **não podem ser embaralhados**: embaralhar deixaria o modelo "ver o futuro". Por isso o corte é por tempo: treino = parte mais antiga, teste = parte mais recente.
Também removemos do treino as linhas cujo alvo cairia dentro do período de teste (faixa de `HORIZONTE_MIN` antes do corte), e o `StandardScaler`/`OneHotEncoder` são ajustados **só no treino** (dentro do `Pipeline`).

In [ ]:
ts_unicos = np.sort(base["ts"].unique())
corte = pd.Timestamp(ts_unicos[int(len(ts_unicos) * PROP_TREINO)])
treino = base[base["ts"] < corte - pd.Timedelta(minutes=HORIZONTE_MIN)]
teste = base[base["ts"] >= corte]
print(f"corte em {corte} | treino: {len(treino)} linhas | teste: {len(teste)} linhas")

X_tr, X_te = treino[NUM + CAT], teste[NUM + CAT]
y_tr_reg, y_te_reg = treino["espera_futura"], teste["espera_futura"]
y_tr_clf, y_te_clf = treino["espera_longa"], teste["espera_longa"]
print("espera longa no treino:", round(y_tr_clf.mean(), 3), "| no teste:", round(y_te_clf.mean(), 3))
assert y_tr_clf.nunique() == 2, "Treino sem as duas classes: ajuste LIMIAR_LONGA ou colete mais dados."

def preprocessador():
    return ColumnTransformer([("num", StandardScaler(), NUM),
                              ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])

## 6. Regressão linear — prever `espera_futura` (minutos)

In [ ]:
reg = Pipeline([("pre", preprocessador()), ("m", LinearRegression())])
reg.fit(X_tr, y_tr_reg)
pred = np.clip(reg.predict(X_te), 0, None)

mae = mean_absolute_error(y_te_reg, pred)
rmse = np.sqrt(mean_squared_error(y_te_reg, pred))
r2 = r2_score(y_te_reg, pred)

# baseline ingênuo: "daqui a 15 min vai estar igual a agora"
pred_base = X_te["espera_atual"]
mae_b = mean_absolute_error(y_te_reg, pred_base)
rmse_b = np.sqrt(mean_squared_error(y_te_reg, pred_base))
r2_b = r2_score(y_te_reg, pred_base)

print(pd.DataFrame({"Regressão linear": [mae, rmse, r2], "Baseline (igual a agora)": [mae_b, rmse_b, r2_b]},
                   index=["MAE (min)", "RMSE (min)", "R²"]).round(3))

nomes = reg.named_steps["pre"].get_feature_names_out()
coefs = pd.Series(reg.named_steps["m"].coef_, index=nomes).sort_values(key=abs, ascending=False)
print("\nVariáveis de maior peso (coeficientes padronizados):")
print(coefs.head(10).round(3))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(y_te_reg, pred, s=6, alpha=0.4)
lim = [0, max(y_te_reg.max(), pred.max())]
ax[0].plot(lim, lim, "r--"); ax[0].set_xlabel("real (min)"); ax[0].set_ylabel("previsto (min)")
ax[0].set_title("Real vs previsto (teste)")
coefs.head(10)[::-1].plot.barh(ax=ax[1]); ax[1].set_title("Top 10 coeficientes")
plt.tight_layout(); plt.show()

## 7. Regressão logística — probabilidade de espera longa

In [ ]:
clf = Pipeline([("pre", preprocessador()),
                ("m", LogisticRegression(max_iter=2000, class_weight="balanced"))])
clf.fit(X_tr, y_tr_clf)
proba = clf.predict_proba(X_te)[:, 1]
pred_c = (proba >= 0.5).astype(int)

cm = confusion_matrix(y_te_clf, pred_c)
print("Matriz de confusão (linhas=real, colunas=previsto):\n", cm)
print("Precisão:", round(precision_score(y_te_clf, pred_c, zero_division=0), 3))
print("Recall  :", round(recall_score(y_te_clf, pred_c, zero_division=0), 3))
print("F1      :", round(f1_score(y_te_clf, pred_c, zero_division=0), 3))
if y_te_clf.nunique() == 2:
    print("ROC AUC :", round(roc_auc_score(y_te_clf, proba), 3))

ConfusionMatrixDisplay(cm, display_labels=["curta", "longa"]).plot(cmap="Blues")
plt.title(f"Espera longa = >= {LIMIAR_LONGA} min, em {HORIZONTE_MIN} min"); plt.show()

nomes_c = clf.named_steps["pre"].get_feature_names_out()
odds = pd.Series(np.exp(clf.named_steps["m"].coef_[0]), index=nomes_c).sort_values(ascending=False)
print("\nMaiores razões de chance (odds ratio, variáveis padronizadas):")
print(odds.head(8).round(2))

## 7.1 Exportar os modelos para o app Fila Certa

Gera `modelo.json` (intercepto, coeficientes e padronizador das duas regressões). O app em Java aplica esses números às leituras do painel de Barueri para mostrar a previsão em 15 min e a chance de espera longa.

In [ ]:
# 7.1 - Exportar os modelos para o app Fila Certa (Java)
# Os dois modelos sao lineares: o app so precisa do intercepto, dos coeficientes e do padronizador.
import json

def exportar_modelo(caminho="modelo.json"):
    pre = reg.named_steps["pre"]
    scaler = pre.named_transformers_["num"]
    ohe = pre.named_transformers_["cat"]
    cats = {c: [str(v) for v in valores] for c, valores in zip(CAT, ohe.categories_)}
    lr, lg = reg.named_steps["m"], clf.named_steps["m"]
    n_esperado = len(NUM) + sum(len(v) for v in cats.values())
    assert len(lr.coef_) == n_esperado and len(lg.coef_[0]) == n_esperado, "ordem/tamanho das variaveis mudou"
    saida = {
        "versao": 1,
        "fonte_treino": "real" if MODO == "real" else "SIMULADO",
        "gerado_em": str(pd.Timestamp.now().floor("s")),
        "horizonte_min": HORIZONTE_MIN,
        "limiar_longa": LIMIAR_LONGA,
        "num": NUM,
        "scaler": {"mean": scaler.mean_.tolist(), "scale": scaler.scale_.tolist()},
        "cat": cats,
        "reg": {"intercept": float(lr.intercept_), "coef": lr.coef_.tolist()},
        "clf": {"intercept": float(lg.intercept_[0]), "coef": lg.coef_[0].tolist()},
        "metricas": {
            "mae_min": float(mae), "mae_baseline_min": float(mae_b), "r2": float(r2),
            "precisao": float(precision_score(y_te_clf, pred_c, zero_division=0)),
            "recall": float(recall_score(y_te_clf, pred_c, zero_division=0)),
            "f1": float(f1_score(y_te_clf, pred_c, zero_division=0)),
            "auc": float(roc_auc_score(y_te_clf, proba)) if y_te_clf.nunique() == 2 else 0.0,
        },
    }
    Path(caminho).write_text(json.dumps(saida, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Modelo exportado em", caminho, "| treino:", saida["fonte_treino"])
    print("Copie para src/main/resources/ml/modelo.json (ou aponte filacerta.ml.modelo-path para ele).")
    return saida

modelo_exportado = exportar_modelo()


## 8. Recomendações baseadas nas previsões

Aplicamos os modelos ao **último instante observado** para responder a pergunta do cidadão da Fila Digital: *"onde vale a pena ir agora?"*

In [ ]:
ultimo_ts = feats.dropna(subset=["espera_passada"])["ts"].max()
agora = feats[(feats["ts"] == ultimo_ts)].dropna(subset=["espera_passada"]).copy()
agora["previsto_min"] = np.clip(reg.predict(agora[NUM + CAT]), 0, None).round(0)
agora["prob_longa"] = clf.predict_proba(agora[NUM + CAT])[:, 1].round(2)
print("Estado em:", ultimo_ts)

def recomendar(setor, top=3):
    t = agora[agora["setor"] == setor].sort_values(["previsto_min", "prob_longa"])
    return t[["local", "espera_atual", "previsto_min", "prob_longa"]].head(top).reset_index(drop=True)

for s in ["CLINICO", "PEDIATRIA", "ORTOPEDIA"]:
    if (agora["setor"] == s).any():
        print(f"\nMelhores opções agora — {s}:")
        print(recomendar(s))

sobrecarga = agora[agora["prob_longa"] >= 0.5].sort_values("prob_longa", ascending=False)
print("\nSetores com alta chance de espera longa nos próximos", HORIZONTE_MIN, "min:")
print(sobrecarga[["local", "espera_atual", "previsto_min", "prob_longa"]].head(8).to_string(index=False)
      if len(sobrecarga) else "nenhum no momento")

### Ações recomendadas (preencher com os números do seu resultado)

1. **Orientar o cidadão no app:** mostrar a espera prevista (não só a atual) e sugerir a unidade/setor com menor previsão entre as opções do mesmo serviço.
2. **Alerta de sobrecarga:** quando `prob_longa` passar de um limite (ex.: 0,5) em um setor, avisar o gestor para remanejar equipe.
3. **Horários de pico:** usar o gráfico de espera por hora para sugerir aos cidadãos casos não urgentes fora do pico.

### Limitações (seja honesto na entrega)

- Os dados são **tempo médio de espera agregado** por setor, não registros individuais: o modelo prevê o painel da prefeitura, não a experiência de cada pessoa.
- Cobrem **uma cidade** e o período coletado; se faltar variedade de horários e dias, os coeficientes de hora e fim de semana não generalizam.
- Eventos raros (surtos, epidemias, feriados) quase não aparecem no treino.
- A espera atual já explica grande parte da futura; o ganho sobre o baseline é o que deve ser discutido, com os números da seção 6.
- Classes desbalanceadas na logística: por isso usamos `class_weight="balanced"` e olhamos recall e F1, não só acurácia.
- Correlação não é causalidade: os coeficientes descrevem associação.

## 9. Arquivos da entrega

In [ ]:
integrantes = '''Nome completo - RM
(preencher)
'''
Path("integrantes.txt").write_text(integrantes, encoding="utf-8")
print("Gerado: integrantes.txt")
print("CSV usado:", CSV_REAL if MODO == "real" else "espera_simulado.csv (SIMULADO - substituir pelo real)")
# Entregar em um ZIP: este .ipynb (executado) + integrantes.txt + CSV + indicação da fonte (seção 'Fonte dos dados').

# 10. Integração com a API de Dados Abertos do Ministério da Saúde (DEMAS)

Esta seção complementa o projeto Fila Digital com dados públicos oficiais para estudar a demanda nas UBS do município de São Paulo.

**Correções desta versão:**
- A URL-base das requisições é `https://apidadosabertos.saude.gov.br` (sem `/v1`). O `/v1/` é o caminho da interface Swagger, não o prefixo das rotas de dados.
- A paginação usa `offset` como número da página, começando em zero. Avançamos de 1 em 1, não somando o tamanho da página.
- A rota de UBS documenta somente `limit` e `offset`; não inventamos filtro municipal.
- SIAPS aceita `municipio`; SIA/SUS aceita `co_ibge`.
- PNI 2026 documenta paginação, mas não filtro municipal. Sua coleta fica como amostra opcional, sem cruzamento automático por UBS.

**Limitação metodológica:** dados DEMAS são registros históricos/administrativos. Não representam fila em tempo real nem tempo de espera individual. Para prever filas atuais, o projeto ainda precisa de dados operacionais com data/hora de chegada, início/fim do atendimento e tipo de serviço.

In [ ]:
# 10.0 - Configuração e endpoints documentados
import requests
import pandas as pd
import time
from pathlib import Path

# IMPORTANTE: não adicionar /v1 nesta URL-base.
BASE_DEMAS = "https://apidadosabertos.saude.gov.br"
ENDPOINTS = {
    "ubs_cnes": "/assistencia-a-saude/unidade-basicas-de-saude",
    "sia_producao": "/assistencia-a-saude/sia-procedimentos-ambulatoriais",
    "siaps_atendimento": "/atencao-primaria/siaps-atendimento-individual",
    "pni_2026": "/vacinacao/doses-aplicadas-pni-2026",
}
CODIGO_IBGE_SP = "3550308"
PASTA_DEMAS = Path("dados_demas")
PASTA_DEMAS.mkdir(exist_ok=True)

# Comece com 2 páginas para validar. Depois de conferir os dados, altere para None.
MAX_PAGINAS_TESTE = 2
SESSION = requests.Session()
SESSION.headers.update({"Accept": "application/json", "User-Agent": "FilaDigital-DataScience/1.0"})


def _extrair_registros(payload):
    """Normaliza respostas JSON em uma lista de registros."""
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        for chave in ("data", "dados", "items", "results", "content", "registros"):
            valor = payload.get(chave)
            if isinstance(valor, list):
                return valor
        return [payload] if payload else []
    raise ValueError(f"Formato JSON inesperado: {type(payload).__name__}")


def consultar_pagina_demas(endpoint, params=None, limit=1000, offset=0, timeout=90):
    """Consulta uma página. offset é o número da página: 0, 1, 2..."""
    parametros = dict(params or {})
    parametros["limit"] = min(int(limit), 1000)
    parametros["offset"] = int(offset)
    url = BASE_DEMAS.rstrip("/") + endpoint
    resposta = SESSION.get(url, params=parametros, timeout=timeout)
    if not resposta.ok:
        raise requests.HTTPError(
            f"HTTP {resposta.status_code} em {resposta.url}. Resposta: {resposta.text[:500]}",
            response=resposta,
        )
    try:
        payload = resposta.json()
    except ValueError as erro:
        raise ValueError(f"A API não retornou JSON em {resposta.url}: {resposta.text[:500]}") from erro
    return pd.DataFrame(_extrair_registros(payload))


def coletar_demas(endpoint, params=None, limit=1000, max_paginas=None, pausa=0.15):
    """Percorre páginas sucessivas; offset é incrementado de 1 em 1 conforme Swagger."""
    limite = min(int(limit), 1000)
    partes = []
    pagina = 0
    while True:
        df_pagina = consultar_pagina_demas(endpoint, params=params, limit=limite, offset=pagina)
        if df_pagina.empty:
            break
        partes.append(df_pagina)
        print(f"Página {pagina}: {len(df_pagina)} registros")
        pagina += 1
        if max_paginas is not None and pagina >= max_paginas:
            break
        if len(df_pagina) < limite:
            break
        if pausa:
            time.sleep(pausa)
    if not partes:
        return pd.DataFrame()
    return pd.concat(partes, ignore_index=True).drop_duplicates().reset_index(drop=True)


def salvar_demas(df, nome):
    caminho = PASTA_DEMAS / f"{nome}.csv"
    df.to_csv(caminho, index=False, encoding="utf-8-sig")
    print(f"Arquivo salvo: {caminho} | {len(df):,} registros")
    return caminho


def mostrar_colunas(df, nome):
    print(f"\n{nome}: {len(df):,} linhas e {len(df.columns)} colunas")
    print(df.columns.tolist())
    display(df.head(5))


## 10.1 - Teste de conexão

Esta célula consulta somente um registro por endpoint para confirmar que as rotas respondem. Se algum serviço falhar, a mensagem exibirá o status HTTP e parte da resposta da API.

In [ ]:
# Teste rápido das quatro rotas - apenas 1 registro por consulta
for nome, endpoint in ENDPOINTS.items():
    try:
        teste = consultar_pagina_demas(endpoint, limit=1, offset=0)
        print(f"OK - {nome}: {len(teste)} registro(s); colunas: {teste.columns.tolist()}")
    except Exception as erro:
        print(f"FALHA - {nome}: {erro}")


## 10.2 - Coleta dos dados principais

- CNES/UBS: lista de UBS cadastradas. A rota não documenta filtro municipal; filtraremos São Paulo após inspecionar as colunas.
- SIA/SUS: filtro pelo código IBGE do município (`co_ibge`).
- SIAPS: filtro pelo código IBGE do município (`municipio`).

A primeira execução coleta duas páginas por fonte. Depois de validar os retornos, altere `MAX_PAGINAS_TESTE = None` na seção 10.0 para percorrer todas as páginas disponíveis.

In [ ]:
# 1) Cadastro de UBS - sem filtro municipal documentado nesta rota
ubs_cnes_br = coletar_demas(
    ENDPOINTS["ubs_cnes"], limit=1000, max_paginas=MAX_PAGINAS_TESTE
)
mostrar_colunas(ubs_cnes_br, "UBS - retorno nacional (amostra)")

# 2) Produção ambulatorial SIA/SUS - São Paulo (filtro documentado: co_ibge)
sia_sp = coletar_demas(
    ENDPOINTS["sia_producao"],
    params={"co_ibge": int(CODIGO_IBGE_SP)},
    limit=1000, max_paginas=MAX_PAGINAS_TESTE
)
mostrar_colunas(sia_sp, "SIA/SUS - São Paulo (amostra)")

# 3) Atendimento individual SIAPS - São Paulo (filtro documentado: municipio)
siaps_sp = coletar_demas(
    ENDPOINTS["siaps_atendimento"],
    params={"municipio": CODIGO_IBGE_SP},
    limit=1000, max_paginas=MAX_PAGINAS_TESTE
)
mostrar_colunas(siaps_sp, "SIAPS - São Paulo (amostra)")


## 10.3 - Filtrar UBS de São Paulo após inspecionar os campos

A rota `/assistencia-a-saude/unidade-basicas-de-saude` documenta somente `limit` e `offset`; ela não documenta `codigo_municipio` nem `co_ibge`. A célula procura nomes comuns para o código IBGE. Se não encontrar uma coluna compatível, não faz um filtro arriscado: mantém os dados em `ubs_cnes_br` e pede inspeção manual.

In [ ]:
colunas_municipio_possiveis = [
    "codigo_ibge", "co_ibge", "codigo_municipio", "co_municipio",
    "cod_municipio", "municipio_ibge", "codigo_municipio_ibge"
]
coluna_municipio = next(
    (c for c in colunas_municipio_possiveis if c in ubs_cnes_br.columns), None
)

if coluna_municipio:
    valores = ubs_cnes_br[coluna_municipio].astype(str).str.replace(r"\.0$", "", regex=True)
    ubs_cnes_sp = ubs_cnes_br[valores == CODIGO_IBGE_SP].copy()
    print(f"Filtro aplicado por {coluna_municipio}. Registros na amostra: {len(ubs_cnes_sp)}")
else:
    ubs_cnes_sp = pd.DataFrame()
    print("Coluna de código IBGE não identificada automaticamente.")
    print("Confira as colunas listadas na seção 10.2 antes de filtrar manualmente.")


## 10.4 - PNI (vacinação): amostra opcional

O Swagger do endpoint PNI 2026 documenta paginação, mas não documenta filtro por município ou CNES. A célula baixa somente uma página de amostra. Não percorra toda a base nem faça merge por UBS antes de verificar se a resposta contém CNES e qual é a granularidade dos registros.

Existem endpoints anuais PNI de 2020 a 2026; os filtros e campos variam por ano e devem ser conferidos individualmente.

In [ ]:
pni_amostra = coletar_demas(
    ENDPOINTS["pni_2026"], limit=1000, max_paginas=1
)
mostrar_colunas(pni_amostra, "PNI 2026 - amostra nacional")


## 10.5 - Salvar os dados coletados

Os CSVs serão salvos na pasta `dados_demas` do ambiente do notebook. As amostras servem para validar a estrutura. Para obter todos os registros, configure `MAX_PAGINAS_TESTE = None` e execute novamente as seções de coleta e salvamento.

In [ ]:
salvar_demas(ubs_cnes_br, "ubs_cnes_amostra")
salvar_demas(sia_sp, "sia_sp_amostra")
salvar_demas(siaps_sp, "siaps_sp_amostra")
salvar_demas(pni_amostra, "pni_2026_amostra")
if not ubs_cnes_sp.empty:
    salvar_demas(ubs_cnes_sp, "ubs_cnes_sao_paulo_amostra")


## 10.6 - Integração com Machine Learning

**Chaves e granularidade:**
- Antes do merge, confirme que SIA/SUS e SIAPS retornam o código CNES (`co_cnes`, `codigo_cnes` ou equivalente) e a competência temporal.
- Preserve CNES como texto para não perder zeros à esquerda.
- Padronize competência como `AAAAMM` e agregue por `CNES + competência` antes de cruzar as fontes.
- No PNI, só faça cruzamento por UBS se houver código CNES no retorno. Sem isso, use como indicador regional/municipal somente se houver identificação geográfica confiável.

**Variáveis candidatas:** atendimentos individuais SIAPS, quantidade produzida SIA/SUS, categoria profissional, classificação CID/CIAP-2, tipo de procedimento, características cadastrais da UBS e mês/ano.

**Separação dos serviços:** não classifique automaticamente consulta especializada, vacinação ou atendimento espontâneo apenas pelo nome da fonte. Use códigos e descrições oficiais após inspecionar o dicionário e os campos retornados.

**Alvo do modelo:** para prever demanda mensal, pode-se usar o total mensal de atendimentos/procedimentos por UBS, desde que a definição seja consistente. Para prever fila/tempo de espera, é necessário outro conjunto operacional com horários e tempos observados. Os dados DEMAS, sozinhos, não fornecem fila em tempo real.

**Atenção ao notebook original:** o modelo anterior usa dados de Prontos-Atendimentos de Barueri. Não misture essa série diretamente com UBS de São Paulo. Crie uma base e uma validação específicas para o novo recorte geográfico e tipo de unidade.

In [ ]:
# 10.7 - Inspeção dos campos para preparar o merge (sem assumir nomes)
for nome, df in {
    "UBS/CNES": ubs_cnes_br,
    "SIA/SUS": sia_sp,
    "SIAPS": siaps_sp,
    "PNI": pni_amostra,
}.items():
    print(f"\n{nome} | shape={df.shape}")
    print(df.columns.tolist())
